# Import Train Data, Drop Columns w Missing & Create Train/Valid datasets

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

housing_training = pd.read_csv("train.csv", index_col="Id")

X = housing_training.drop("SalePrice", axis=1)
y = housing_training["SalePrice"]

columns_with_too_many_missings = [col for col in X.columns if X[col].isnull().sum()/X.shape[0] > 0.05]
X.drop(columns_with_too_many_missings, axis=1, inplace=True)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.2, random_state = 0)

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

# 

# Create Baseline Score Function to evaluate data treatments

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def baseline_model(X_train, y_train, X_valid, y_valid):
    baseline_model = RandomForestRegressor(n_estimators=100, random_state=0)
    baseline_model.fit(X_train, y_train)
    predictions = baseline_model.predict(X_valid)
    return mean_absolute_error(y_valid, predictions)

# Drop categorical columns to get a base line score

In [5]:
print(baseline_model(X_train.select_dtypes(exclude=['object']), y_train, X_valid.select_dtypes(exclude=['object']), y_valid))

18199.780502283105


# Create Numerical Simple Imputer to median

In [6]:
from sklearn.impute import SimpleImputer

numerical_imputer = SimpleImputer(strategy='median')
numerical_columns = [col for col in X_train.columns if X_train[col].dtype != 'object']
#X_train_numerical_cols_w_missings = [col for col in numerical_columns if X_train[col].isnull().any()]
print(numerical_columns) 
#X_train[X_train_numerical_cols_w_missings] = numerical_imputer.fit_transform(X_train[X_train_numerical_cols_w_missings])


['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']


# Create Categorical Pipeline - Change to not exclude any categorical column and see how that goes (low hopes given the concentration of records in a single value of these columns)

In [7]:
# Analyze categorical columns and underlying unique values
categorical_columns = [col for col in X_train.columns if X_train[col].dtype == 'object']

#Identify categorical columns for which X_valid values are a subset of X_train, and only keep them
not_subset_categorical_column = [col for col in categorical_columns if not(X_valid[col].isin(X_train[col]).all())]
print(not_subset_categorical_column)
#Drop these columns from the exercise for now
X_train.drop(not_subset_categorical_column, axis=1, inplace=True)
X_valid.drop(not_subset_categorical_column, axis=1, inplace=True)


categorical_columns = list(set(categorical_columns) - set(not_subset_categorical_column))

X_train.shape

['Condition2', 'RoofMatl', 'Functional']


(1168, 64)

In [8]:
# Define what columns should be looked at from ordinal and one hot encoding standpoint
high_cardinality_columns = [col for col in categorical_columns if X_train[col].nunique() > 10]
low_cardinality_columns = [col for col in categorical_columns if X_train[col].nunique() <= 10]

print(len(high_cardinality_columns), len(low_cardinality_columns))

3 27


In [9]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# Create ordinal encoding for high cardinality columns
categorical_ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)

#Create one hot encoding for low cardinality columns
categorical_one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

In [10]:
#Transform X_train and X_valid datasets using above imputers


#Categorical Ordinal Imputation
#X_train[high_cardinality_columns] = categorical_ordinal_encoder.fit_transform(X_train[high_cardinality_columns])
#X_valid[high_cardinality_columns] = categorical_ordinal_encoder.transform(X_valid[high_cardinality_columns])   


#Categorical One Hot Encoding Imputation
def one_hot_encoder(X, columns_to_impute, fit = False):
    
    print("X Shape Before One Hot: ", X.shape)    
    print("Number of impute columns: ", len(columns_to_impute))
    print("Sum of X colums to impute nuniques : ", X[columns_to_impute].nunique().sum())
    
    if fit:
        categorical_one_hot_encoder.fit(X[columns_to_impute])

    X_one_hot_encoder_output = categorical_one_hot_encoder.transform(X[columns_to_impute])
    X_one_hot_encoder_output = pd.DataFrame(X_one_hot_encoder_output, columns=categorical_one_hot_encoder.get_feature_names_out())
    X_one_hot_encoder_output.index = X.index
    
    X.drop(columns_to_impute, axis=1, inplace=True)
    X = pd.concat([X, X_one_hot_encoder_output], axis = 1)   
    
    print("X  Shape After One Hot: ", X.shape)

    return X


#X_train = one_hot_encoder(X_train, low_cardinality_columns, fit = True)
#X_valid = one_hot_encoder(X_valid, low_cardinality_columns)

print(X_train.shape)
print(X_valid.shape)

(1168, 64)
(292, 64)


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

categorical_ordinal_pipeline = Pipeline(steps=[
    ('ordinalimpute', SimpleImputer(strategy="most_frequent")),
    ('ordinal', categorical_ordinal_encoder) ])

categorical_one_hot_pipeline = Pipeline(steps=[
    ('onehotimpute', SimpleImputer(strategy="most_frequent")),
    ('onehot', categorical_one_hot_encoder)])

categorical_transformer = ColumnTransformer(transformers=[
    ('ordinal', categorical_ordinal_pipeline, high_cardinality_columns),
    ('onehot', categorical_one_hot_pipeline, low_cardinality_columns),
    ('numerical', numerical_imputer, numerical_columns)])

print("Pre X Train Shape: ", X_train.shape)
print("Pre X Valid Shape: ", X_valid.shape)

X_train = pd.DataFrame(categorical_transformer.fit_transform(X_train), columns = categorical_transformer.get_feature_names_out())
X_valid = pd.DataFrame(categorical_transformer.transform(X_valid), columns=categorical_transformer.get_feature_names_out())

print("Pos X Train Shape: ", X_train.shape)
print("Pos X Valid Shape: ", X_valid.shape)


Pre X Train Shape:  (1168, 64)
Pre X Valid Shape:  (292, 64)
Pos X Train Shape:  (1168, 169)
Pos X Valid Shape:  (292, 169)


# Get baseline model score after imputation

In [12]:
print(baseline_model(X_train, y_train, X_valid, y_valid))


17276.62421232877


# Investigate columns that were dropped and that were present in X train for pottential benefits

In [13]:
original_train_dataset = housing_training.drop("SalePrice", axis=1)
dropped_columns = []
for col_original in original_train_dataset.columns:
    dropped_column = True
    for col_X_train in X_train.columns:
        if col_original in col_X_train:
            dropped_column = False
            break
    if dropped_column:
        dropped_columns.append(col_original)
            

print("Columns dropped: ", dropped_columns, "\n Number of columns dropped :", len(dropped_columns)) 

X_unexplored_features = housing_training[dropped_columns]
columns_w_missings = []
columns_not_subset = []
other_columns = []
for col in X_unexplored_features.columns:
    if X_unexplored_features[col].isnull().sum()/X_unexplored_features.shape[0] > 0.005:
        #print(col, " - column dropped due too many missing values: ", X_unexplored_features[col].isnull().sum()/X_unexplored_features.shape[0])
        columns_w_missings.append(col)
    elif X_unexplored_features[col].dtype == 'object' :
        #print(col, " - categorical column dropped due to train values not being subset of valid ones", X_unexplored_features[col].value_counts(dropna=False))
        columns_not_subset.append(col)
    else:
        other_columns.append(col)

print("Columns with too many missing values: ", columns_w_missings, "Number :", len(columns_w_missings)) 
print("Columns not subset: ", columns_not_subset, "Number: ", len(columns_not_subset))  
print("Other columns: ", other_columns, "Number: ", len(other_columns))



#print([X_unexplored_features[col].value_counts(dropna=False) for col in X_unexplored_features.select_dtypes(include=['object'])])

Columns dropped:  ['LotFrontage', 'Alley', 'Condition2', 'RoofMatl', 'MasVnrType', 'Functional', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'] 
 Number of columns dropped : 15
Columns with too many missing values:  ['LotFrontage', 'Alley', 'MasVnrType', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'] Number : 12
Columns not subset:  ['Condition2', 'RoofMatl', 'Functional'] Number:  3
Other columns:  [] Number:  0


# Study columns not subset to identify potential to include any of them in the data treatment pipeline

In [14]:
print([X_unexplored_features[col].value_counts(dropna=False) for col in X_unexplored_features[columns_not_subset].columns])

#Included these columns in the dataset but not worth it because it didn't improve the score compared to the added complexity (given the onehotcode approach)

[Condition2
Norm      1445
Feedr        6
Artery       2
RRNn         2
PosN         2
PosA         1
RRAn         1
RRAe         1
Name: count, dtype: int64, RoofMatl
CompShg    1434
Tar&Grv      11
WdShngl       6
WdShake       5
Metal         1
Membran       1
Roll          1
ClyTile       1
Name: count, dtype: int64, Functional
Typ     1360
Min2      34
Min1      31
Mod       15
Maj1      14
Maj2       5
Sev        1
Name: count, dtype: int64]


# Study columns dropped due to the number of missings

In [15]:
X_columns_w_missings = housing_training[columns_w_missings] 
print(X_columns_w_missings.info())

#change the columns with missing threshold from 0.5% missings to 5% missing values, and that improved significantly the score


<class 'pandas.core.frame.DataFrame'>
Index: 1460 entries, 1 to 1460
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   LotFrontage   1201 non-null   float64
 1   Alley         91 non-null     object 
 2   MasVnrType    588 non-null    object 
 3   FireplaceQu   770 non-null    object 
 4   GarageType    1379 non-null   object 
 5   GarageYrBlt   1379 non-null   float64
 6   GarageFinish  1379 non-null   object 
 7   GarageQual    1379 non-null   object 
 8   GarageCond    1379 non-null   object 
 9   PoolQC        7 non-null      object 
 10  Fence         281 non-null    object 
 11  MiscFeature   54 non-null     object 
dtypes: float64(2), object(10)
memory usage: 148.3+ KB
None


# Study outlier potentially required treatment for kept numerical columns

In [16]:
import seaborn as sns

originally_numerical_columns = [col for col in X_train.columns if col.startswith("numerical_")]
print("Numer of numerical cols: ", len(originally_numerical_columns), "\nList: ", originally_numerical_columns)

def detect_outliers_zscore(data, column_name, thres = 3):
    outliers = []
    mean = np.mean(data[column_name])
    std = np.std(data[column_name])
    #print(mean, std)
    for i in range(len(data[column_name])):
        z_score = (data[column_name][i]-mean)/std
        if (np.abs(z_score) > thres):
            outliers.append(i)
    return outliers# Driver code

def remove_numerical_outliers(data, target_data, numerical_columns, approach = "replace", strategy = "median", n_columns_for_id_to_be_removed = 2):
    columns_outliers = {}
    agg_outliers_indexs = []
    for col in numerical_columns:
        outliers = detect_outliers_zscore(data, col)
        columns_outliers[col] = outliers
        agg_outliers_indexs += outliers

    outliers_indexes = list(dict.fromkeys(agg_outliers_indexs))
    indexes_to_review = []

    #Check the indexes and the number of columns they represent an outlier to
    for index in outliers_indexes:
        counter = agg_outliers_indexs.count(index)
        if counter >= n_columns_for_id_to_be_removed:
            indexes_to_review.append(index)

    print("Number of indexes to review: ", len(indexes_to_review)) 
    if approach == "remove":
        return data.drop(indexes_to_review, inplace=False), target_data.drop(indexes_to_review, inplace=False)
    elif approach == "replace":
        data_copy = data.copy()
        for index in indexes_to_review:
            for col in numerical_columns:
                if index in columns_outliers[col]:
                    data_copy.loc[index, col] = data_copy[col].median()
        return data_copy, target_data
    else:
        print("Error, approach not recognized")


#Tried different approaches to handle these outliers but without success, the score was always worse than the original one. Therefore this function invokation in following code block is commented

Numer of numerical cols:  34 
List:  ['numerical__MSSubClass', 'numerical__LotArea', 'numerical__OverallQual', 'numerical__OverallCond', 'numerical__YearBuilt', 'numerical__YearRemodAdd', 'numerical__MasVnrArea', 'numerical__BsmtFinSF1', 'numerical__BsmtFinSF2', 'numerical__BsmtUnfSF', 'numerical__TotalBsmtSF', 'numerical__1stFlrSF', 'numerical__2ndFlrSF', 'numerical__LowQualFinSF', 'numerical__GrLivArea', 'numerical__BsmtFullBath', 'numerical__BsmtHalfBath', 'numerical__FullBath', 'numerical__HalfBath', 'numerical__BedroomAbvGr', 'numerical__KitchenAbvGr', 'numerical__TotRmsAbvGrd', 'numerical__Fireplaces', 'numerical__GarageCars', 'numerical__GarageArea', 'numerical__WoodDeckSF', 'numerical__OpenPorchSF', 'numerical__EnclosedPorch', 'numerical__3SsnPorch', 'numerical__ScreenPorch', 'numerical__PoolArea', 'numerical__MiscVal', 'numerical__MoSold', 'numerical__YrSold']


# Random Forest Regressor - Hyperparameter optimization

Start with Random Search to get a sense of the magnitude of the best hyperparameter selection

In [17]:
from sklearn.model_selection import RandomizedSearchCV
n_estimators = [int(x) for x in np.linspace(start = 2500, stop = 5000, num = 100)]
max_features = ['sqrt']
max_depth = [int(x) for x in np.linspace(10, 200, num = 10)]
min_samples_leaf = [2, 3, 4]
min_samples_split = [2, 3, 4]
bootstrap = [True, False]

random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_leaf': min_samples_leaf,
               'min_samples_split': min_samples_split,
               'bootstrap': bootstrap}

#random_forest = RandomForestRegressor()
#random_forest_hyperparameter_optimization = RandomizedSearchCV(estimator = random_forest, param_distributions = random_grid, n_iter = 100, cv = 3, verbose = 2, random_state = 0, n_jobs = -1)
#random_forest_hyperparameter_optimization.fit(X_train, y_train)

#print(random_forest_hyperparameter_optimization.best_params_)


Use the knowledge obtained of the magnitude of the best hyperparameters to get the perfect ones using GridSearchCV

In [18]:
#Apply knowledge obtained on best parameters by having more tuned in hyperparameters to serve as input to GridSearchCV
from sklearn.model_selection import GridSearchCV

n_estimators = [2600, 2650 , 2700, 2750]
max_features = ['sqrt']
max_depth = [130, 140, 150]
min_samples_leaf = [2,3]
min_samples_split = [2,3]
bootstrap = [False]

param_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_leaf': min_samples_leaf,
               'min_samples_split': min_samples_split,
               'bootstrap': bootstrap}

#grid = GridSearchCV(RandomForestRegressor(), param_grid=param_grid, cv =  3, n_jobs = -1, verbose = 2)
#grid.fit(X_train, y_train)

#print(grid.best_params_)
#best_model = grid.best_estimator_


# Use Random and Grid Search Search to find the best model and underlying hypermarameters (instead of focusing on random forest alone)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

"""
model_definition_grid = {
    'linear_regression': {
        'model': SVR(gamma='auto'),
        'params': {
            'C': [5, 7], #identified that 1 and 10 have pratically the same score, despite 10 taking way too long
            'kernel': ['linear']
        }
    },
    'ridge': {
        'model': Ridge(),
        'params': {
            'alpha': [0.001, 100] #try out 0.001 and 1e-15 afterwards
        }
    },   
    'elastic_net': {
        'model': ElasticNet(max_iter = 10000),
        'params':{
            'alpha': [0.1, 1], #try out 10 afterwards
            'l1_ratio': [0.4, 0.7], #try out 0.1 or 1
            'tol': [0.001]
            }
    },
    'svr': {
        'model': SVR(),
        'params':{
            'epsilon': [0.1, 0.5],
            'C': [1, 10],
            'kernel': ['rbf']
        }
    },
    'gradient_boosting': {
        'model': GradientBoostingRegressor(),
        'params':{
            'n_estimators': [200, 400],
            'learning_rate':[0.1, 0.5],
            'max_depth':[3, 5]
        }
    },
    'nearest_neighbors': {
        'model': KNeighborsRegressor(),
        'params':{
            'n_neighbors': [2, 5, 7, 10]
        }
    },
    'random_forest': {
        'model': RandomForestRegressor(),
        'params':{                            #using best params found on the previous iteration of the exercise (see above block)
            'n_estimators': [2650],
            'max_depth': [150],
            'min_samples_split': [3],
            'min_samples_leaf': [2],
            'max_features': ['sqrt'],
            'bootstrap':[False]
        }
    }
}

scores = []
for model_name, model_parameters in model_definition_grid.items():
    model_grid = GridSearchCV(model_parameters['model'], model_parameters['params'], cv = 3, verbose = 3)
    model_grid.fit(X_train, y_train)
    
    predictions = model_grid.best_estimator_.predict(X_valid)
    mae = mean_absolute_error(y_valid, predictions)

    print(model_name, " Best MAE: ", mae)

    scores.append({
        'model': model_name,
        'best_params': model_grid.best_params_,
        'best_score': model_grid.best_score_,
        'best_valid_mae': mae
    })
"""

Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV 1/3] END ................C=5, kernel=linear;, score=0.763 total time= 1.6min
[CV 2/3] END ................C=5, kernel=linear;, score=0.783 total time=  36.3s
[CV 3/3] END ................C=5, kernel=linear;, score=0.803 total time=  23.9s
[CV 1/3] END ................C=7, kernel=linear;, score=0.768 total time= 5.5min
[CV 2/3] END ................C=7, kernel=linear;, score=0.788 total time= 1.4min
[CV 3/3] END ................C=7, kernel=linear;, score=0.806 total time= 1.2min
linear_regression  Best MAE:  24970.331339896202
Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV 1/3] END .......................alpha=0.001;, score=0.764 total time=   0.0s
[CV 2/3] END .......................alpha=0.001;, score=0.874 total time=   0.0s
[CV 3/3] END .......................alpha=0.001;, score=0.871 total time=   0.0s
[CV 1/3] END .........................alpha=100;, score=0.810 total time=   0.0s
[CV 2/3] END ..........

c:\Users\User\anaconda3\envs\WorkSpace\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.389e+11, tolerance: 4.885e+09
  model = cd_fast.enet_coordinate_descent(


[CV 1/3] END alpha=0.1, l1_ratio=0.4, tol=0.001;, score=0.809 total time=   1.4s


c:\Users\User\anaconda3\envs\WorkSpace\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.740e+11, tolerance: 4.243e+09
  model = cd_fast.enet_coordinate_descent(


[CV 2/3] END alpha=0.1, l1_ratio=0.4, tol=0.001;, score=0.874 total time=   1.7s


...

[CV 2/3] END ..alpha=1, l1_ratio=0.7, tol=0.001;, score=0.857 total time=   1.7s


c:\Users\User\anaconda3\envs\WorkSpace\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.701e+11, tolerance: 5.235e+09
  model = cd_fast.enet_coordinate_descent(


[CV 3/3] END ..alpha=1, l1_ratio=0.7, tol=0.001;, score=0.867 total time=   1.5s


c:\Users\User\anaconda3\envs\WorkSpace\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.919e+11, tolerance: 7.191e+09
  model = cd_fast.enet_coordinate_descent(


elastic_net  Best MAE:  20433.47300053137
Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV 1/3] END .....C=1, epsilon=0.1, kernel=rbf;, score=-0.034 total time=   0.1s
[CV 2/3] END .....C=1, epsilon=0.1, kernel=rbf;, score=-0.109 total time=   0.2s
[CV 3/3] END .....C=1, epsilon=0.1, kernel=rbf;, score=-0.021 total time=   0.1s
[CV 1/3] END .....C=1, epsilon=0.5, kernel=rbf;, score=-0.034 total time=   0.1s
[CV 2/3] END .....C=1, epsilon=0.5, kernel=rbf;, score=-0.109 total time=   0.1s
[CV 3/3] END .....C=1, epsilon=0.5, kernel=rbf;, score=-0.021 total time=   0.1s
[CV 1/3] END ....C=10, epsilon=0.1, kernel=rbf;, score=-0.032 total time=   0.1s
[CV 2/3] END ....C=10, epsilon=0.1, kernel=rbf;, score=-0.109 total time=   0.1s
[CV 3/3] END ....C=10, epsilon=0.1, kernel=rbf;, score=-0.018 total time=   0.1s
[CV 1/3] END ....C=10, epsilon=0.5, kernel=rbf;, score=-0.032 total time=   0.1s
[CV 2/3] END ....C=10, epsilon=0.5, kernel=rbf;, score=-0.109 total time=   0.1s
[CV 3/3

Analyze the results from trying out the different models and underlying hyperparameters above

In [64]:
#scores_df = pd.DataFrame(scores)
#print(scores_df)

# best model identified above is grandient boosting - lets do a bit of hyperparameter optimization using the knowledge obtained above
#best_model_to_be_tuned = scores_df.loc[scores_df["best_valid_mae"] == scores_df["best_valid_mae"].min()]["model"]
#best_model_to_be_tuned_params = scores_df.loc[scores_df["best_valid_mae"] == scores_df["best_valid_mae"].min()]["best_params"] #best params from above code block - learning rate 0.1, estimators 400, max_depth 5


#print(best_model_to_be_tuned,best_model_to_be_tuned_params )

#0.07, 5, 200
best_param_grid = {'n_estimators': [200, 300, 500, 1000],
               'max_depth': [3, 5, 7, 15],
               'learning_rate': [0.001, 0.01, 0.05, 0.07, 0.1]}

best_grid = GridSearchCV(GradientBoostingRegressor(), param_grid=best_param_grid, cv =  3, n_jobs = -1, verbose = 2)
best_grid.fit(X_train, y_train)

Fitting 3 folds for each of 80 candidates, totalling 240 fits


GridSearchCV(cv=3, estimator=GradientBoostingRegressor(), n_jobs=-1,
             param_grid={'learning_rate': [0.001, 0.01, 0.05, 0.07, 0.1],
                         'max_depth': [3, 5, 7, 15],
                         'n_estimators': [200, 300, 500, 1000]},
             verbose=2)

In [67]:
best_model = best_grid.best_estimator_
best_params = best_model.get_params #learning_rate = 0.05, n_estimators = 500, 
print("Best Parameters: ", best_params)
best_predictions = best_model.predict(X_valid)
print("Mae from best model/params: ", mean_absolute_error(y_valid, best_predictions))

Best Parameters:  <bound method BaseEstimator.get_params of GradientBoostingRegressor(learning_rate=0.05, n_estimators=500)>
Mae from best model/params:  15219.489468518623


# Export results to submit on Kaggle

In [69]:
#Apply data treatments to the whole X dataset
#X Drop columns with too many missings already dropped initially
X_full = X
y_full = y
X_full = pd.DataFrame(categorical_transformer.fit_transform(X_full), columns = categorical_transformer.get_feature_names_out())
#X_full, y_full = remove_numerical_outliers(X_full, y_full, originally_numerical_columns, approach = "replace", strategy = "mean")
best_model.fit(X_full, y_full)
print("Best model parameters: ", best_model.get_params)
print(best_model.get_params)

#Apply data transformations to X_test
housing_testing = pd.read_csv("test.csv", index_col="Id")
X_test = pd.DataFrame(categorical_transformer.transform(housing_testing), columns = categorical_transformer.get_feature_names_out())
X_test.index = housing_testing.index
predicts = best_model.predict(X_test)
print(predicts)

output = pd.DataFrame({'Id': X_test.index,
                       'SalePrice': predicts})
output.to_csv('kaggle_housing_submission_3_best_model.csv', index=False)


Best model parameters:  <bound method BaseEstimator.get_params of GradientBoostingRegressor(learning_rate=0.05, n_estimators=500)>
<bound method BaseEstimator.get_params of GradientBoostingRegressor(learning_rate=0.05, n_estimators=500)>
[117091.25123154 161227.87048618 177077.00534209 ... 156936.49399739
 121898.4220485  232055.35726442]
